# Cross-modal PCA/PLS learnable overview

This notebook is an onboarding guide for the learned PCA/PLS family in `models/architectures/crossmodal_pca_pls.py`.

It covers two models:

1. `CrossModal_PCA_PLS_learnable`: PCA encoders, a PLS-initialized latent map, and a PCA decoder exposed as trainable PyTorch parameters.
2. `CrossModal_PCA_PLS_CovProjector`: the same PCA/PLS backbone plus an additive covariate projector in target latent space.

The goal is conceptual clarity plus local dev cells that exercise the real dataloaders, Lightning training path, covariate plumbing, and evaluator path.

In [ ]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import main
import models.registry
import models.train.lightning_module
import models.train.loss
import models.eval.evaluator
import models.architectures.crossmodal_pca_pls
from models.architectures.utils import get_model_input
from models.utils import get_batch_cov

importlib.reload(models.architectures.crossmodal_pca_pls)
importlib.reload(models.train.loss)
importlib.reload(models.train.lightning_module)
importlib.reload(models.eval.evaluator)
importlib.reload(models.registry)
importlib.reload(main)

from main import Sim

RESULTS_ROOT = Path("results/local_results/crossmodal_pca_pls_learnable_overview")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

BASE_SIM_KWARGS = dict(
    source="SC",
    target="FC",
    parcellation="Glasser",
    shuffle_seed=0,
    data_load_mode="precomputed",
)

def show_base_metrics(run_out):
    metrics = run_out["test_metrics"]["base_metrics"]
    return pd.Series(metrics).sort_index()


def inspect_one_batch(sim, model, title):
    batch = next(iter(sim.val_loader))
    x = get_model_input(batch)
    y = batch["y"]
    kwargs = {}
    if getattr(model, "uses_cov", False):
        kwargs["cov"] = get_batch_cov(batch)
    model.eval()
    with torch.no_grad():
        y_hat = model(x, **kwargs) if kwargs else model(x)
    print(title)
    print("x shape:", tuple(x.shape) if torch.is_tensor(x) else {k: tuple(v.shape) for k, v in x.items()})
    print("y shape:", tuple(y.shape))
    print("y_hat shape:", tuple(y_hat.shape))
    print("one-batch mse:", float(torch.mean((y_hat.cpu() - y.cpu()) ** 2)))
    if kwargs:
        print("cov keys:", sorted(kwargs["cov"].keys()))
    return y_hat, y, batch


## Shared data and training path

Learned models use the same `Sim` wrapper, but `learned: true` in their YAML config sends `run_single(...)` through the Lightning training branch.

The local tests below use very small latent dimensions and few epochs. They are smoke tests for onboarding, not tuned experiments.

The data path is:

$$
\texttt{Sim} \rightarrow \texttt{HCP\_Base} \rightarrow
\texttt{DataLoader} \rightarrow \texttt{CrossModalLightningModule} \rightarrow
\texttt{Evaluator}.
$$

In [ ]:
learnable_loader_probe = Sim(
    model_name="CrossModal_PCA_PLS_learnable",
    config_overrides={
        "model": {
            "n_components_pca_source": 32,
            "n_components_pca_target": 32,
            "n_components_pls": 4,
            "device": "cpu",
        },
        "trainer": {"batch_size": 32, "max_epochs": 1, "lr": 1e-4, "loss_type": "mse"},
    },
    **BASE_SIM_KWARGS,
)

print("train batches:", len(learnable_loader_probe.train_loader))
print("val batches:", len(learnable_loader_probe.val_loader))
print("test batches:", len(learnable_loader_probe.test_loader))
probe_batch = next(iter(learnable_loader_probe.train_loader))
print("batch keys:", sorted(probe_batch.keys()))
print("x shape:", tuple(get_model_input(probe_batch).shape))
print("y shape:", tuple(probe_batch["y"].shape))
print("cov keys:", sorted(probe_batch["cov"].keys()))


## 1. `CrossModal_PCA_PLS_learnable`

This model turns the closed-form PCA+PLS pipeline into a PyTorch module.

For each source modality, a fixed or trainable encoder maps source edges into source latent coordinates:

$$
z_{s_i} = (x_{s_i} - \mu_{s_i}) W_{enc,i}.
$$

Source latents are concatenated:

$$
z_{src} = [z_{s_1}; z_{s_2}; \ldots; z_{s_m}].
$$

A middle matrix maps source latents into target latent coordinates:

$$
\hat z_t = \mathrm{dropout}(z_{src}) W_{mid}.
$$

The decoder reconstructs target edges:

$$
\hat y_t = \mu_t + \mathrm{dropout}(\hat z_t) W_{dec}.
$$

With the default `random_init=False`, the pieces are initialized from the closed-form PCA/PLS solution:

- `W_enc`: source PCA loading matrices.
- `W_mid`: PLS regression coefficients in latent space.
- `W_dec`: target PCA loading matrix transposed.

The `learn_encoder`, `learn_mid`, and `learn_decoder` flags decide which pieces are updated by gradient descent.

In [ ]:
learnable_config = {
    "model": {
        "n_components_pca_source": 64,
        "n_components_pca_target": 64,
        "n_components_pls": 8,
        "learn_encoder": False,
        "learn_mid": True,
        "learn_decoder": False,
        "random_init": False,
        "dropout": 0.1,
        "l1_l2_tuple": [0.0, 1.0e-4],
        "device": "cpu",
    },
    "trainer": {
        "max_epochs": 3,
        "batch_size": 32,
        "lr": 1.0e-4,
        "loss_type": "mse",
        "log_every": 1,
    },
}

learnable_sim = Sim(
    model_name="CrossModal_PCA_PLS_learnable",
    config_overrides=learnable_config,
    **BASE_SIM_KWARGS,
)

learnable_run = learnable_sim.run_single(
    mode="dev",
    save_checkpoint=False,
    config_override=learnable_config,
    run_eval=True,
)

show_base_metrics(learnable_run)


In [ ]:
learnable_y_hat, learnable_y, learnable_batch = inspect_one_batch(
    learnable_sim,
    learnable_run["model"],
    "CrossModal_PCA_PLS_learnable one-batch check",
)

model = learnable_run["model"]
print("trainable parameters:", model.get_num_params())
print("W_mid shape:", tuple(model.W_mid.shape))
print("W_dec shape:", tuple(model.W_dec.shape))
print("source encoder shapes:", {k: tuple(v.shape) for k, v in model.source_encoders.items()})


### Latent-loss option

The learned PCA/PLS classes also expose `predict_target_latents(...)` and `encode_target_latents(...)`, so the Lightning module can train directly in target PCA space with `loss_type="latent_mse"` or `loss_type="latent_weighted_mse"`.

For target edges $y_t$,

$$
z_t = (y_t - \mu_t) P_{t,k}.
$$

The latent loss is then

$$
\mathcal{L}_{latent} = \frac{1}{k}\|\hat z_t - z_t\|_2^2.
$$

The default cells use edge-space MSE because it is the simplest onboarding path, but switching the trainer loss type is the intended way to test latent supervision.

In [ ]:
# Optional quick latent-loss smoke test. This is short by design.
latent_loss_config = {
    **learnable_config,
    "trainer": {
        **learnable_config["trainer"],
        "max_epochs": 1,
        "loss_type": "latent_mse",
    },
}

latent_loss_run = learnable_sim.run_single(
    mode="dev",
    save_checkpoint=False,
    config_override=latent_loss_config,
    run_eval=True,
)

show_base_metrics(latent_loss_run)


## 2. `CrossModal_PCA_PLS_CovProjector`

The covariate projector keeps the PCA/PLS backbone but adds an explicit subject-covariate branch in target latent space.

The source pathway is the same backbone:

$$
z_{base} = \mathrm{dropout}(z_{src}) W_{mid}.
$$

Each covariate source is encoded separately:

$$
h_j = g_j(c_j).
$$

The covariate embeddings are concatenated and fused into a target-latent correction:

$$
z_{cov} = f_{cov}([h_1; h_2; \ldots; h_q]).
$$

The final target latent prediction is additive:

$$
\hat z_t = z_{base} + z_{cov}.
$$

The decoder is unchanged:

$$
\hat y_t = \mu_t + \hat z_t W_{dec}.
$$

This model tests whether demographic or FreeSurfer features explain target FC structure beyond what the SC PCA/PLS backbone already captures.

In [ ]:
cov_projector_config = {
    "model": {
        "n_components_pca_source": 64,
        "n_components_pca_target": 64,
        "n_components_pls": 8,
        "learn_encoder": False,
        "learn_mid": True,
        "learn_decoder": False,
        "random_init": False,
        "dropout": 0.1,
        "l1_l2_tuple": [0.0, 1.0e-4],
        "cov_sources": ["age", "sex", "race_eth", "fs_volumes"],
        "cov_projectors": {
            "age": {"type": "linear", "out_dim": 4},
            "sex": {"type": "linear", "out_dim": 4},
            "race_eth": {"type": "linear", "out_dim": 8},
            "fs_volumes": {"type": "linear", "out_dim": 16},
        },
        "cov_fusion": {
            "type": "mlp",
            "hidden_dims": [32],
            "dropout": 0.1,
            "layer_norm": True,
        },
        "use_target_scores_in_projector": False,
        "device": "cpu",
    },
    "trainer": {
        "max_epochs": 3,
        "batch_size": 32,
        "lr": 1.0e-4,
        "loss_type": "mse",
        "log_every": 1,
    },
}

cov_projector_sim = Sim(
    model_name="CrossModal_PCA_PLS_CovProjector",
    config_overrides=cov_projector_config,
    **BASE_SIM_KWARGS,
)

cov_projector_run = cov_projector_sim.run_single(
    mode="dev",
    save_checkpoint=False,
    config_override=cov_projector_config,
    run_eval=True,
)

show_base_metrics(cov_projector_run)


In [ ]:
cov_y_hat, cov_y, cov_batch = inspect_one_batch(
    cov_projector_sim,
    cov_projector_run["model"],
    "CrossModal_PCA_PLS_CovProjector one-batch check",
)

cov_model = cov_projector_run["model"]
with torch.no_grad():
    latent_parts = cov_model.predict_latents(cov_batch)

print("trainable parameters:", cov_model.get_num_params())
print("cov sources:", cov_model.cov_sources)
print("z_base shape:", tuple(latent_parts["z_base"].shape))
print("z_cov shape:", tuple(latent_parts["z_cov"].shape))
print("z_target shape:", tuple(latent_parts["z_target"].shape))


## Compare the learnable smoke-test results

This comparison is meant to verify that both learned paths run through the local training and evaluation stack. It is not a tuned result because the configs intentionally use few epochs.

In [ ]:
learnable_summary = pd.DataFrame({
    "CrossModal_PCA_PLS_learnable": show_base_metrics(learnable_run),
    "CrossModal_PCA_PLS_learnable_latent_mse": show_base_metrics(latent_loss_run),
    "CrossModal_PCA_PLS_CovProjector": show_base_metrics(cov_projector_run),
}).T

learnable_summary


In [ ]:
artifact_dir = RESULTS_ROOT
artifact_dir.mkdir(parents=True, exist_ok=True)
for name, run_out in {
    "CrossModal_PCA_PLS_learnable": learnable_run,
    "CrossModal_PCA_PLS_learnable_latent_mse": latent_loss_run,
    "CrossModal_PCA_PLS_CovProjector": cov_projector_run,
}.items():
    report_base = artifact_dir / f"{name}_eval_test"
    test_eval = run_out["evaluators"]["test"]
    test_eval.analyze_results(
        verbose=False,
        filepath=str(report_base),
        output_format="md",
        model_name=name,
    )
    print(f"wrote {report_base}.md")
